# *Hands-on PEFT without LoRA – Fast & Learner-Friendly Version*

We will use **Prompt Tuning** and **Prefix Tuning** (very memory efficient PEFT methods).

## **Installation**

In [ ]:
# 1. Install dependencies (fast version)
# %pip install -q transformers==4.41.0 peft==0.11.0 datasets==2.20.0 accelerate==0.30.0

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from peft import PromptTuningConfig, PrefixTuningConfig, get_peft_model, TaskType


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\ashwi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\ashwi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

## **Load Dataset**

In [2]:
# 2. Load small dataset subset (IMDB sentiment)
dataset = load_dataset("stanfordnlp/imdb", split="train[:200]")  # Only 200 samples for speed

def preprocess(example):
    return {
        "text": "Review: " + example["text"][:300] + "\nSentiment:",  # truncate for speed
        "labels": example["label"]
    }

dataset = dataset.map(preprocess)
print("Dataset ready:", len(dataset))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset ready: 200


## **Load Model**

In [3]:
# 3. Load model (Frozen base)
model_name = "google/flan-t5-small"   # Much faster than base!

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded!")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded!


## **Prompt Tuning (Recommended for beginners)**

In [4]:
# 4. Apply Prompt Tuning
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,   # Correct for T5!
    num_virtual_tokens=10,             # Small number = very fast
    tokenizer_name_or_path=model_name
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 10,240 || all params: 76,971,392 || trainable%: 0.0133


## **Tokenization**

In [5]:
# 5. Tokenize (fast)
def tokenize(example):
    inputs = tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)
    inputs["labels"] = inputs["input_ids"].copy()
    return inputs

tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

## **Training Stage**

In [6]:
# 6. Train (this should finish in < 5-8 minutes on Colab CPU)
training_args = TrainingArguments(
    output_dir="./prompt_tuning",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,17.312800
20,17.010900
30,16.234400
40,16.836700
50,16.763300


TrainOutput(global_step=50, training_loss=16.83161163330078, metrics={'train_runtime': 196.1161, 'train_samples_per_second': 1.02, 'train_steps_per_second': 0.255, 'total_flos': 9294525235200.0, 'train_loss': 16.83161163330078, 'epoch': 1.0})

## **Testing Stage**

In [7]:
# 7. Save & Test Inference
model.save_pretrained("prompt_tuned_model")
tokenizer.save_pretrained("prompt_tuned_model")

# Quick inference test
inputs = tokenizer("Review: This movie was fantastic!\nSentiment:", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


I've seen this film for a few years and loved it. It was great and the


## ***Alternative Method***

In [8]:
# Reset model and try Prefix Tuning
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

peft_config = PrefixTuningConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    num_virtual_tokens=10
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# You can train the same way as above

trainable params: 81,920 || all params: 77,043,072 || trainable%: 0.1063


## **Simple LangChain Integration (using GPT-2 for speed)**

In [9]:
!pip install -q langchain langchain-community

from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="gpt2",
    tokenizer="gpt2",
    max_new_tokens=50,
    pad_token_id=50256
)

llm = HuggingFacePipeline(pipeline=pipe)

response = llm.invoke("Review: This movie was amazing!\nSentiment:")
print(response)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.5.0 which is incompatible.


/tmp/ipykernel_2882/3432165084.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/tmp/ipykernel_2882/3432165084.py:14: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


Review: This movie was amazing!
Sentiment: We really liked it. The character was fun and hilarious when he was around when he was busy or at work. It was interesting to see how that turned out, because we never knew he would be there during these events. I actually gave it a
